# 1. Imports

In [ ]:
# Import required libraries
import sys
from tqdm import tqdm
from dataclasses import dataclass
import pandas as pd
import numpy as np
from pathlib import Path
import pickle
import json
from collections import defaultdict
from typing import Dict, List, Set, Tuple, Optional, Any
import warnings
warnings.filterwarnings('ignore')
from scipy.spatial.distance import cdist
from scipy.spatial import ConvexHull

import matplotlib.pyplot as plt
import seaborn as sns
import altair as alt
alt.data_transformers.enable("vegafusion")

# GOATOOLS imports for GO data handling
from goatools.obo_parser import GODag
from goatools.anno.gaf_reader import GafReader
from goatools.go_search import GoSearch
from goatools.gosubdag.gosubdag import GoSubDag

sys.path.append("../")
from src.subset_visualization import plot_given_genes_on_feature_space
from matplotlib.backends.backend_pdf import PdfPages
import importlib
importlib.reload(sys.modules['src.subset_visualization'])

plt.style.use("../../config/DIT_HAP.mplstyle")
COLORS = plt.rcParams['axes.prop_cycle'].by_key()['color']
AX_WIDTH, AX_HEIGHT = plt.rcParams['figure.figsize']


# 2. Configuration

In [ ]:
@dataclass
class Config:

    go_obo_file: Path = Path("../../resources/pombase_data/2025-10-01/ontologies_and_associations/go-basic.obo")
    go_gaf_file: Path = Path("../../resources/pombase_data/2025-10-01/ontologies_and_associations/gene_ontology_annotation.gaf.tsv")
    output_dir: Path = Path("../../results/HD_DIT_HAP_generationRAW/24_phenotypic_coherence_analysis")

    DIT_HAP_data_file: Path = Path("../../results/HD_DIT_HAP_generationRAW/18_gene_level_clustering/kmeans_cluster_result.tsv")
    gRNA_data_file: Path = Path("../../resources/260127-all_genes_order1_gRNA_HDdata_fitted_parameters.tsv")
    gene_meta_file: Path = Path("../../resources/pombase_data/2025-10-01/Gene_metadata/gene_IDs_names_products.tsv")

    def __post_init__(self):
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.DIT_HAP_data: pd.DataFrame = pd.read_csv(self.DIT_HAP_data_file, sep="\t")
        self.gene_meta: pd.DataFrame = pd.read_csv(self.gene_meta_file, sep="\t", index_col=1)
        self.gene_meta["gene_name"] = self.gene_meta["gene_name"].fillna(self.gene_meta["gene_systematic_id"])
        self.coding_genes: list = self.gene_meta.query("gene_type == 'protein coding gene'")["gene_systematic_id"].tolist()

        self.gRNA_data: pd.DataFrame = pd.read_csv(self.gRNA_data_file, low_memory=False, sep="\t")
        self.gene_data: pd.DataFrame = pd.merge(self.DIT_HAP_data, self.gRNA_data[["Systematic ID", "um", "lam"]], on="Systematic ID", suffixes=("", "_gRNA"), how="left").set_index("Systematic ID")

cfg = Config()

# 3. Load Go data

In [ ]:
def load_GO_data(obo_file: str | Path, gaf_file: str | Path) -> Tuple[Optional[GODag], Optional[Dict[str, Dict[str, Set[str]]]]]:
    """
    Load GO ontology and gene associations.
    
    Parameters:
    -----------
    obo_file : str | Path
        Path to the OBO file containing ontology definitions
    gaf_file : str
        Path to the GAF file containing gene associations
        
    Returns:
    --------
    Tuple[GODag, Dict[str, Dict[str, Set[str]]]]
        GO ontology DAG and namespace-to-associations dictionary
    """
    try:
        # Load GO ontology
        print(f"Loading GO ontology from: {obo_file}")
        godag = GODag(str(obo_file), optional_attrs=['def', 'relationship'], load_obsolete=False)
        print(f"Loaded {len(godag)} GO terms")
        
        # Load gene associations
        print(f"Loading gene associations from: {gaf_file}")
        gaf_reader = GafReader(str(gaf_file), godag = godag)
        
        # Group associations by namespace
        ns2assoc = gaf_reader.get_ns2assc(godag=godag)
            
        return godag, gaf_reader, ns2assoc
        
    except Exception as e:
        print(f"Error loading GO data: {e}")
        return None, None, None


# Load GO data and create comprehensive mapping
print("=" * 60)
print("LOADING GO DATA AND CREATING COMPREHENSIVE MAPPING")
print("=" * 60)

# Load GO ontology and gene associations
godag, gaf_reader, ns2assoc = load_GO_data(cfg.go_obo_file, cfg.go_gaf_file)
gene2go = gaf_reader.get_id2gos_nss(propagate_counts=True, relationships={'part_of', 'is_a'}, load_obsolete=False)
go2genes = gaf_reader.get_id2gos_nss(go2geneids=True, propagate_counts=True, relationships={'part_of', 'is_a'}, load_obsolete=False)

# 4. Create the dataframe with genes as the index and GO terms as the columns

In [ ]:
def go_details(go_id: str, go_dag: GODag) -> dict:
    """
    Get the details of a GO term from the GO DAG.
    """
    go_term = go_dag[go_id]

    return {
        "id": go_id,
        "name": go_term.name,
        "namespace": go_term.namespace,
        "definition": go_term.defn,
        "level": go_term.level,
        "depth": go_term.depth,
    }

def dit_hap_coverage_of_go_term(term_genes: list, gene_data: pd.DataFrame) -> dict:
    """
    Calculate the DIT-HAP coverage of a GO term.
    """
    term_genes = gene_data[gene_data.index.isin(term_genes)].copy()
    term_genes_count = len(term_genes)
    term_genes_clusters = sorted(term_genes["revised_cluster"].unique().tolist())
    return {
        "term_genes_count": term_genes_count,
        "term_genes_clusters": ",".join(map(str, term_genes_clusters)),
    }

In [ ]:
go_table = pd.DataFrame()

for i, (go_id, genes) in enumerate(tqdm(go2genes.items())):
    go_detail = go_details(go_id, godag)
    codings = sorted([ g for g in genes if g in cfg.coding_genes ])
    go_detail["Coding_genes"] = ",".join(codings)
    go_detail["Coding_genes_count"] = len(codings)
    dit_hap_coverage = dit_hap_coverage_of_go_term(codings, cfg.gene_data)
    go_detail["covered_genes_count"] = dit_hap_coverage["term_genes_count"]
    go_detail["covered_genes_clusters"] = dit_hap_coverage["term_genes_clusters"]
    go_table = pd.concat([go_table, pd.DataFrame([go_detail])], ignore_index=True)

In [ ]:
go_table

# 5. Calculate phenotypic coherence for each GO term

In [ ]:
def geometric_median(X, epsilon=1e-5):
    """
    计算 Geometric Median 使用 Weiszfeld 算法
    X: (N, 2) 的 numpy 数组
    """
    # 1. 初始猜测：使用普通的坐标中位数 (计算快，作为起点很准)
    y = np.median(X, axis=0)
    
    while True:
        # 计算当前中心点 y 到所有点 X[i] 的欧几里得距离
        # cdist 返回 shape (1, N)，我们需要 (N,)
        distances = cdist(X, [y]).flatten()
        
        # 处理距离为0的情况（防止除以0错误）
        # 如果中心点恰好落在某个数据点上，给一个极小值
        distances = np.where(distances == 0, 1e-10, distances)
        
        # 计算权重：距离的倒数 (离得越远，权重越小，把中心拉回来的力越小)
        weights = 1.0 / distances
        
        # 计算新的加权平均中心
        y_next = np.sum(X * weights[:, np.newaxis], axis=0) / np.sum(weights)
        
        # 检查是否收敛 (位置变化是否小于阈值)
        if np.linalg.norm(y - y_next) < epsilon:
            break
            
        y = y_next
        
    return y

In [ ]:
scaled_gene_data = cfg.gene_data.copy()
scaled_gene_data["um"] = scaled_gene_data["um"].apply(lambda x: x if x < 1.3 else 1.3)
scaled_gene_data["lam"] = scaled_gene_data["lam"].apply(lambda x: x/10)

for idx, row in go_table.iterrows():
    term = row["name"]
    gene_ids = row["Coding_genes"].split(",")
    covered_genes_count = row["covered_genes_count"]
    if covered_genes_count == 0:
        print(f"Skipping GO term '{term}' with 0 covered genes.")
        continue
    gene_coords = scaled_gene_data[scaled_gene_data.index.isin(gene_ids)][["um", "lam"]].to_numpy()
    gm = geometric_median(gene_coords)
    distances = cdist(gene_coords, [gm]).flatten()
    mean_absolute_distance = np.mean(distances)
    median_absolute_distance = np.median(distances)

    # Calculate Convex Hull area if there are at least 3 covered genes
    try:
        hull = ConvexHull(gene_coords)
        area = hull.volume
    except Exception as e:
        area = 0

    go_table.at[idx, "geometric_median_um"] = gm[0]
    go_table.at[idx, "geometric_median_lam"] = gm[1]
    go_table.at[idx, "mean_absolute_distance"] = mean_absolute_distance
    go_table.at[idx, "median_absolute_distance"] = median_absolute_distance
    go_table.at[idx, "convex_hull_area"] = area

# 6. Scatter plot

In [ ]:
alt.Chart(go_table.query("geometric_median_um > 0.2")).mark_point().encode(
    x="mean_absolute_distance",
    y="convex_hull_area",
    tooltip=go_table.columns.tolist()
) | alt.Chart(go_table.query("geometric_median_um > 0.2")).mark_point().encode(
    x="mean_absolute_distance",
    y="median_absolute_distance",
    tooltip=go_table.columns.tolist()
)

In [ ]:
coordinated_terms = go_table.query("mean_absolute_distance < 0.1 and convex_hull_area < 0.1 and geometric_median_um > 0.2 and covered_genes_count >=3").sort_values(["covered_genes_count"], ascending=False)

In [ ]:
with PdfPages(cfg.output_dir / "coordinated_GO_terms_gene_distributions.pdf") as pdf:
    for idx, row in tqdm(coordinated_terms.iterrows(), total=len(coordinated_terms)):
        term = row["name"]
        gene_ids = row["Coding_genes"].split(",")

        fig, axes = plt.subplots(1, 2, figsize=(12, 6))
        plot_given_genes_on_feature_space(
            ax=axes[0],
            data_df=cfg.gene_data.reset_index(),
            genes=gene_ids,
            gene_column="Systematic ID",
            title=term
        )
        plot_given_genes_on_feature_space(
            ax=axes[1],
            data_df=cfg.gene_data.reset_index(),
            genes=gene_ids,
            gene_column="Systematic ID",
            title=f"{term}\n(Scaled Feature Space)",
            x_feature="um_gRNA",
            y_feature="lam_gRNA",
        )
        for i in range(2):
            axes[i].set_xlim(-0.3, 1.8)
            axes[i].set_ylim(-0.5, 14)

        plt.tight_layout()
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

In [ ]:
id2name = dict(zip(cfg.gene_meta["gene_systematic_id"], cfg.gene_meta["gene_name"]))
coordinated_terms["gene_names"] = coordinated_terms["Coding_genes"].apply(lambda x: ",".join([ id2name[g] for g in x.split(",") ]))

coordinated_terms.sort_values("gene_names").to_csv(cfg.output_dir / "coordinated_GO_terms_summary.tsv", sep="\t", index=False)

In [ ]:
annotated_coordinated_terms = pd.read_excel(cfg.output_dir / "coordinated_GO_terms_summary_manual_selected.xlsx", sheet_name="Sheet1")

In [ ]:
fig, axes = plt.subplots(3,4,figsize=(24, 18))
axes = axes.flatten()

for idx, row in tqdm(annotated_coordinated_terms.iloc[:12].iterrows(), total=len(annotated_coordinated_terms)):
    term = row["name"]
    if len(term) > 30:
        # wrap long titles
        term_elements = term.split(" ")
        sep_idx = len(term_elements) // 2
        term = " ".join(term_elements[:sep_idx]) + "\n" + " ".join(term_elements[sep_idx:])
    gene_ids = row["Coding_genes"].split(",")
    ax = axes[idx]
    plot_given_genes_on_feature_space(
        ax=ax,
        data_df=cfg.gene_data.reset_index(),
        genes=gene_ids,
        gene_column="Systematic ID",
        title=term,
        cmap=COLORS[idx%8]
    )

    ax.set_xlim(-0.3, 1.8)
    ax.set_ylim(-0.5, 14)
    ax.set_xlabel("DR")
    ax.set_ylabel("DL")

plt.tight_layout()
plt.show()
plt.close(fig)

In [ ]:
def plot_given_genes_on_feature_space(
    ax,
    data_df: pd.DataFrame,
    genes: list[str],
    gene_column: str,
    title: str,
    x_feature: str = "um",
    y_feature: str = "lam",
    cmap: str = 'viridis',
    label: str = "Selected Genes",
    title_with_count: bool = True,
    **kwargs
):
    """ Plot given genes on feature space. """

    # all points in light gray
    x_all = data_df[x_feature]
    y_all = data_df[y_feature]
    ax.scatter(x_all, y_all, color='lightgray', alpha=0.4, **kwargs)

    # points for given genes
    subset_df = data_df.query(f"`{gene_column}` in @genes")
    x_subset = subset_df[x_feature]
    y_subset = subset_df[y_feature]
    try:
        xy_subset = np.vstack([x_subset, y_subset])
        # ax.scatter(x_subset, y_subset, c=z, cmap=cmap, **kwargs, label=f"{label} (n={len(subset_df)})")
        ax.scatter(x_subset, y_subset, color=cmap, **kwargs, label=f"{label} (n={len(subset_df)})", alpha=0.7)
    except Exception:
        ax.scatter(x_subset, y_subset, color='red', **kwargs, label=f"{label} (n={len(subset_df)})")
    if title_with_count:
        ax.set_title(f'{title}\n(n={len(subset_df)})')
    else:
        ax.set_title(f'{title}')
    ax.set_xlabel(x_feature)
    ax.set_ylabel(y_feature)
    # ax.grid(True)

    return ax

In [ ]:
x_feature = "um"
y_feature = "lam"
with PdfPages(cfg.output_dir / "coordinated_GO_terms_with_category.pdf") as pdf:
    for category, category_df in annotated_coordinated_terms.groupby("Category"):
        fig, axes = plt.subplots(1, 2, figsize=(16, 7))
        category_df = category_df.reset_index()
        data_df = cfg.gene_data.reset_index()
        x_all = data_df[x_feature]
        y_all = data_df[y_feature]
        for col, col_suffix in enumerate(["", "_gRNA"]):
            ax = axes[col]
            ax.set_xlim(-0.3, 1.8)
            ax.set_ylim(-0.5, 14)
            ax.set_title(category)
            ax.scatter(x_all, y_all, color='lightgray', alpha=0.4)
            for idx, row in category_df.iterrows():
                term = row["name"]
                gene_ids = row["Coding_genes"].split(",")
                feature_x = x_feature + col_suffix
                feature_y = y_feature + col_suffix
                ax = axes[col]
                subset_df = data_df.query(f"`Systematic ID` in @gene_ids")
                x_subset = subset_df[feature_x]
                y_subset = subset_df[feature_y]
                ax.scatter(x_subset, y_subset, alpha=0.7, label=f"{term} (n={len(subset_df)})")
            
            ax.legend(loc='upper right', fontsize='small')

        plt.tight_layout()
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)